In [1]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from useful_functions import *
import pandas as pd
from joblib import Parallel, delayed

In [2]:
spectra_data    = fits.open('/Users/hyp0515/data/0715_Spring_BGS_ALL_trimmed.fits')
color_data      = fits.open('/Users/hyp0515/data/0715_Spring_half_BGS_BRIGHT_catalog_with_Flux.fits')
cigale_data     = fits.open('/Users/hyp0515/data/IronPhysProp_v1.2_extracted.fits')
fastspecfit     = fits.open('/Users/hyp0515/data/0715_Spring_half_BGS_BRIGHT_catalog_fastspecfit.fits')

ids = read_ids('dp_samples_ids.txt')

SPECTRA = Spectrum(spectra_data, color_data, cigale_data, fastspecfit, load_targetID=ids)


In [3]:
# SPECTRA = SPECTRA.subtype_filter(subtype='QSO', exclude=True)
# SPECTRA.shrink_dataset(50)
SPECTRA.stack_data()
SPECTRA.mask_bad()

In [4]:
FIT     = FitSpectrum()
SPECTRA = FIT.shift_to_rest_frame(SPECTRA)
SPECTRA = FIT.label_emission_lines(SPECTRA, 3)
print(SPECTRA.n_spectra)


20716


In [5]:
# SPECTRA = FIT.significant_emission_filter(SPECTRA)
# print(SPECTRA.n_spectra)

In [6]:
SPECTRA.df.head(5)

,TARGETID,RA,DEC,SPECTYPE,FLUX_G,FLUX_R,FLUX_Z,LOGM,LOGSFR,z_pipe,z,OII,Hbeta,OIII,Halpha,NII,SII
0,39627739380056564,177.019312,-1.965308,GALAXY,8.337815,20.474783,36.728188,10.452706,0.941032,0.323187,0.323187,False,False,True,True,True,False
1,39627739380057506,177.058476,-1.984356,GALAXY,30.548021,72.780373,140.431030,10.324258,-0.880754,0.102379,0.102379,False,False,False,True,True,False
2,39627739380058467,177.097982,-1.888290,GALAXY,37.576733,74.504120,134.410553,10.402426,1.031611,0.123931,0.123931,True,False,False,True,True,True
3,39627739380060210,177.177947,-1.988386,GALAXY,11.878757,29.588015,52.399445,10.496312,0.039087,0.238389,0.238389,False,True,False,True,True,True
4,39627739380061198,177.216425,-1.948593,GALAXY,22.125759,44.621117,80.619362,9.514435,0.903058,0.103277,0.103277,False,False,False,True,True,True


In [7]:
def process_target(target_id):
    """
    Processes a single target to find double-peaked features.
    """
    # try:
    p_value, delta_dv, dp_detection, line_fluxes_rank, params_2comp = FIT.find_dp(SPECTRA, id=target_id)

    dp_cols = ['OII3726_dp', 'OII3729_dp',
            'Hbeta_dp',
            'OIII4959_dp', 'OIII5007_dp',
            'NII6548_dp', 'Halpha_dp', 'NII6583_dp', 
            'SII6716_dp', 'SII6731_dp']

    dp_rank_cols = [f'{col[:-3]}_rank' for col in dp_cols]

    data = {
        'TARGETID': target_id,
        'p_value': p_value,
        'dv': params_2comp['dv'],
        'delta_dv': delta_dv,
        'sigma': params_2comp['sigma'],
    }
    data.update(dict(zip(dp_cols, dp_detection)))
    data.update(dict(zip(dp_rank_cols, line_fluxes_rank)))
    return data
    # except Exception as e:
    #     print(f"Error processing target {target_id}: {e}")
    #     return None

    

# Use joblib to parallelize the processing over all target IDs
# n_jobs=-1 uses all available CPU cores.
results = Parallel(n_jobs=10)(delayed(process_target)(target_id) for target_id in tqdm(SPECTRA.targetID))

# Convert the list of dictionaries to a DataFrame
dp_df = pd.DataFrame(results)

# display(dp_df)

100%|██████████| 20716/20716 [01:23<00:00, 247.05it/s]


In [8]:
display(dp_df.head(10))

,TARGETID,p_value,dv,delta_dv,sigma,OII3726_dp,OII3729_dp,Hbeta_dp,OIII4959_dp,OIII5007_dp,...,OII3726_rank,OII3729_rank,Hbeta_rank,OIII4959_rank,OIII5007_rank,NII6548_rank,Halpha_rank,NII6583_rank,SII6716_rank,SII6731_rank
0,39627739380056564,1.238700e-05,"(56.426444360169135, -76.68395634380273)",133.110401,"(56.34585781136056, 42.60295818685627)",False,False,False,False,False,...,-1,-1,-1,4,2,3,0,1,-1,-1
1,39627739380057506,1.980505e-04,"(120.66957743140641, -57.412754275781644)",178.082332,"(51.710374006710055, 80.13838572456841)",False,False,False,False,False,...,-1,-1,-1,-1,-1,2,0,1,-1,-1
2,39627739380058467,3.273379e-03,"(87.12450939570255, -28.974182203031084)",116.098692,"(45.01100642275467, 60.64196361432912)",False,True,False,False,False,...,3,1,-1,-1,-1,6,0,2,4,5
3,39627739380060210,3.079846e-06,"(80.43325908228452, -86.82168409201246)",167.254943,"(60.72488335053829, 65.94087199970755)",False,False,False,False,False,...,-1,-1,2,-1,-1,4,0,1,3,5
4,39627739380061198,1.110223e-16,"(88.49232105348224, -72.96380752674673)",161.456129,"(49.931878048942906, 61.28260876843729)",False,False,False,False,False,...,-1,-1,-1,-1,-1,3,0,1,2,4
5,39627739380062010,2.867254e-02,"(56.974564382951186, -21.72862166348826)",78.703186,"(52.593007803654814, 74.98688630156494)",False,False,False,False,False,...,-1,-1,-1,-1,-1,4,0,1,2,3
6,39627739384251743,4.276592e-02,"(46.733397407536515, -112.69685037579943)",159.430248,"(99.31925606858547, 32.15227190283853)",False,False,False,False,False,...,-1,-1,-1,-1,-1,2,0,1,-1,-1
7,39627739384252917,1.171390e-05,"(54.86337125674715, -69.48611675832838)",124.349488,"(44.181646596378414, 60.56577398491257)",False,False,False,False,False,...,-1,-1,2,-1,-1,3,0,1,-1,-1
8,39627739384253866,5.570100e-12,"(74.85144350501935, -84.00981187606229)",158.861255,"(69.10275153135159, 57.86563716199026)",False,False,False,False,False,...,-1,-1,1,-1,-1,4,0,2,3,5
9,39627739384254171,4.658520e-03,"(110.84054650547823, -21.97295288717676)",132.813499,"(30.000000000005706, 86.68931868720975)",False,False,False,False,False,...,-1,-1,-1,-1,-1,2,0,1,-1,-1


In [9]:
dp_df.to_csv('dp_cand_results.csv', index=False)

In [10]:
dp_df = pd.read_csv('dp_cand_results.csv', low_memory=False)
print(f'All: {len(dp_df)}')
dp_candidates = dp_df[(dp_df['p_value'] < 0.05) & (dp_df['delta_dv'] > 75)]
print(f'DP Candidates: {len(dp_candidates)} ({len(dp_candidates) / len(dp_df) * 100:.2f}%)')

All: 20716
DP Candidates: 20716 (100.00%)
